# Визуализация — мини-проект

Сквозной мини-проект: один датасет, решения передаются между этапами. См. docs/project_notebook.md.


In [ ]:
# Практика к уроку — выполняйте ячейки по порядку
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)



## Постановка задачи `(intro)`


Один датасет **Titanic** — последовательно отвечаем на вопросы графиками. Каждый раздел опирается на выводы предыдущего; в конце — простая модель и ROC.


## Загрузка данных `(eda)`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

%matplotlib inline
np.random.seed(42)
sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"]
df["Embarked"] = df["embarked"].astype(str)
print(df.shape)



## Вопрос 1: как распределены Age и Fare? `(viz)`


**Вывод:** Age — почти нормальный хвост; Fare — сильный перекос, нужен boxplot по классам.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["Age"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Age")
axes[0].set_title("Histogram возраста")
sns.boxplot(data=df, x="Pclass", y="Fare", hue="Pclass", ax=axes[1], palette="Blues", legend=False)
axes[1].set_title("Boxplot Fare по классу")
plt.tight_layout()
plt.show()



## Вопрос 2: связаны ли Age, Fare и выживание? `(viz)`


**Вывод:** выжившие чаще в 1-м классе и с более высоким Fare; scatter подтверждает.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for survived, color, label in [(0, "crimson", "не выжил"), (1, "steelblue", "выжил")]:
    sub = df[df["Survived"] == survived]
    axes[0].scatter(sub["Age"], sub["Fare"], alpha=0.35, s=15, c=color, label=label)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Fare")
axes[0].legend()
axes[0].set_title("Scatter Age vs Fare")

rates = df.groupby("Pclass")["Survived"].mean()
axes[1].bar(rates.index.astype(str), rates.values, color="steelblue")
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("Pclass")
axes[1].set_ylabel("доля выживших")
axes[1].set_title("Bar: survival by class")
plt.tight_layout()
plt.show()



## Вопрос 3: корреляции числовых признаков? `(viz)`


**Вывод:** Survived слабо коррелирует с Pclass (−) и Fare (+) — согласуется с barplot выше.


In [ ]:
num = df[["Survived", "Pclass", "Age", "Fare"]].dropna()
corr = num.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Корреляции")
plt.tight_layout()
plt.show()



## Вопрос 4: роль Sex и порта? `(viz)`


**Вывод:** женщины выживали чаще; Cherbourg — выше доля выживших (согласуется с 1-м классом).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="Sex", hue="Sex", ax=axes[0], palette="Set2", legend=False)
sns.barplot(data=df, x="Sex", y="Survived", hue="Sex", estimator="mean", ax=axes[1], palette="muted", legend=False)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("доля выживших")
fig.suptitle("Категориальные plot")
plt.tight_layout()
plt.show()



## Вопрос 5: сводная панель EDA `(viz)`


**Решение:** собираем ключевые графики в одну фигуру 2×2 — итог истории EDA.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].hist(df["Fare"].dropna(), bins=40, color="steelblue", edgecolor="white")
axes[0, 0].set_title("hist Fare")
axes[0, 1].scatter(df["Age"], df["Fare"], c=df["Survived"], alpha=0.3, s=10, cmap="coolwarm")
axes[0, 1].set_title("scatter c=Survived")
df["Survived"].value_counts().plot(kind="bar", ax=axes[1, 0], color=["crimson", "steelblue"])
axes[1, 0].set_title("bar Survived")
sns.boxplot(data=df, x="Sex", y="Age", hue="Sex", ax=axes[1, 1], palette="pastel", legend=False)
axes[1, 1].set_title("box Age by Sex")
plt.tight_layout()
plt.show()



## Вопрос 6: насколько предсказуемо выживание? `(viz)`


По EDA: Pclass, Fare, Sex — сильные предикторы. **Решение:** простая LogReg; один split; ROC на том же test.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay

df_ml = df[["Pclass", "Age", "Fare", "Survived"]].copy()
X = df_ml.drop(columns=["Survived"])
y = df_ml["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

final_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)
final_pipe.fit(X_train, y_train)
proba = final_pipe.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
residual_proxy = y_test.values - proba
axes[0].scatter(proba, residual_proxy, alpha=0.5)
axes[0].axhline(0, color="crimson", ls="--")
axes[0].set_xlabel("P(survive)")
axes[0].set_ylabel("y - proba")
axes[0].set_title("Диагностика вероятностей")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
plt.tight_layout()
plt.show()



## Чек-лист `(summary)`


1. Один `df` — вопросы и графики **по порядку**.
2. Каждый раздел — вывод, следующий на него опирается.
3. Bar: ось Y с нуля; heatmap для corr.
4. Subplots 2×2 — итог EDA.
5. После гипотез — модель и ROC на **том же** test.
